# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the available record sets in the dataset and list their `@id` fields, together with the available fields in each record set, referencing all of them by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # Single field as dict
        fields = [fields]
    print("  Field @ids:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field['@id']}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# In this dataset, let's extract all record sets and convert them to DataFrames
import pprint

# Retrieve record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Reading records from RecordSet @id='" + record_set_id + "'")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with columns: {df.columns.tolist()}\n")
        print(df.head())
    else:
        print(f"No records returned for RecordSet {record_set_id}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

_Choose fields referenced by their `@id` for each operation. Adjust the field `@ids` below as appropriate for the dataset contents._

In [ ]:
# Select one record set and perform EDA
if dataframes:
    # Use first loaded record set as example
    first_record_set_id = list(dataframes.keys())[0]
    df = dataframes[first_record_set_id]
    print(f"Exploring record set: {first_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Try to find a numeric field (example: columns containing 'abundance', 'MW', or 'count')
    numeric_fields = []
    for col in df.columns:
        if df[col].dtype in [int, float] or df[col].apply(lambda x: isinstance(x, (int, float))).all():
            numeric_fields.append(col)
    if not numeric_fields:
        # Try casting possible columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if df[col].dtype in [int, float]:
                    numeric_fields.append(col)
            except (ValueError, TypeError):
                pass
    print(f"Numeric fields detected: {numeric_fields}\n")

    if numeric_fields:
        # Pick first as example
        numeric_field = numeric_fields[0]  # In place of <numeric_field_id>
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 0
        print(f"Filtering for {numeric_field} > {threshold:.2f}\n")
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records:")
        print(filtered_df.head())

        # Normalization
        filtered_df[numeric_field + "_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Try grouping by another column (e.g., 'description' or next non-numeric field)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA analysis.")
else:
    print("No DataFrames were loaded, cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram of a numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    df = dataframes[first_record_set_id]
    # Find a numeric field
    numeric_fields = [
        col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])
    ]
    if not numeric_fields:
        # Try to convert
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_fields = [
            col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])
        ]
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {field} in RecordSet @id '{first_record_set_id}'")
        plt.xlabel(field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No DataFrames available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to programmatically access dataset metadata, records, and fields by their `@id` using the `mlcroissant` library.
- DataFrames for record sets were loaded, and basic EDA and visualization were performed on numeric fields, referenced by their `@id`s.
- You can further extend this notebook by referencing additional record sets or fields, and by applying specific analyses relevant to mass spectrometry or proteomics data.